<a href="https://colab.research.google.com/github/RSL161/Data-Analysis/blob/master/NLP_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Сделать классификацию данных fakenews
Используя ноутбук занятия (также размещен в папке Materials) и данные fakenews, 3 раза разными способами получить на задаче классификации значение f1 выше 0.91 для методов на sklearn и выше 0.52 для методов на pytorch.

In [1]:
import pandas as pd
import numpy as np

### Project_1

In [2]:
!wget https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv

--2021-11-22 19:31:54--  https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1253562 (1.2M) [text/plain]
Saving to: ‘Constraint_Train.csv.2’

Constraint_Train.cs 100%[===================>]   1.20M  --.-KB/s    in 0.05s   

2021-11-22 19:31:55 (23.1 MB/s) - ‘Constraint_Train.csv.2’ saved [1253562/1253562]



In [3]:
df = pd.read_csv('Constraint_Train.csv')

In [4]:
df.head()

,id,tweet,label
0,1,The CDC currently reports 99031 deaths. In gen...,real
1,2,States reported 1121 deaths a small rise from ...,real
2,3,Politically Correct Woman (Almost) Uses Pandem...,fake
3,4,#IndiaFightsCorona: We have 1524 #COVID testin...,real
4,5,Populous states can generate large case counts...,real


In [5]:
from nltk.tokenize import sent_tokenize, word_tokenize

In [6]:
from tqdm import tqdm

In [7]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
sentences = [word_tokenize(text.lower()) for text in tqdm(df.tweet)]

100%|██████████| 6420/6420 [00:02<00:00, 2773.06it/s]


In [9]:
from gensim.models.word2vec import Word2Vec
%time model_tweets = Word2Vec(sentences, workers=3, size=300, min_count=4, window=3)

CPU times: user 4 s, sys: 51.5 ms, total: 4.06 s
Wall time: 2.67 s


In [10]:
model_tweets.wv.most_similar('covid')

[('pandas', 0.9804303646087646),
 ('nashville', 0.978183388710022),
 ('dranthonyfauci', 0.9772359132766724),
 ('beer', 0.9753706455230713),
 ('socialdistancing', 0.9726893901824951),
 ('nba', 0.97217857837677),
 ('coronaupdatesinindia', 0.9720569849014282),
 ('unemployed', 0.9719012975692749),
 ('facemasks', 0.9682123064994812),
 ('complications', 0.9667243957519531)]

In [11]:
model_tweets.init_sims()

In [12]:
def get_text_embedding(text):
    result = []
    for word in word_tokenize(text.lower()):
        if word in model_tweets.wv:
            result.append(model_tweets.wv[word])

    if len(result):
        result = np.sum(result, axis=0)
    else:
        result = np.zeros(300)
    return result

In [13]:
features = [get_text_embedding(text) for text in tqdm(df.tweet)]

100%|██████████| 6420/6420 [00:03<00:00, 1979.61it/s]


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(features, df.label, test_size=0.33)

In [16]:
model = LogisticRegression()
model.fit(X_train, y_train)

/usr/local/lib/python3.7/dist-packages/sklearn/linear_model/_logistic.py:818: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  extra_warning_msg=_LOGISTIC_SOLVER_CONVERGENCE_MSG,


LogisticRegression()

In [17]:
from sklearn.metrics import classification_report

In [18]:
predicted = model.predict(X_test)

In [19]:
print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

        fake       0.86      0.88      0.87       998
        real       0.89      0.87      0.88      1121

    accuracy                           0.88      2119
   macro avg       0.88      0.88      0.88      2119
weighted avg       0.88      0.88      0.88      2119



### Project_2

In [1]:
import pandas as pd
import numpy as np

In [2]:
!wget https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv

--2021-11-22 19:05:12--  https://raw.githubusercontent.com/diptamath/covid_fake_news/main/data/Constraint_Train.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1253562 (1.2M) [text/plain]
Saving to: ‘Constraint_Train.csv’

Constraint_Train.cs 100%[===================>]   1.20M  --.-KB/s    in 0.05s   

2021-11-22 19:05:13 (23.2 MB/s) - ‘Constraint_Train.csv’ saved [1253562/1253562]



In [3]:
df = pd.read_csv('Constraint_Train.csv')

In [7]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [8]:
from nltk.corpus import stopwords
stop = set(stopwords.words('english'))

In [9]:
df['tweet'] = df['tweet'].apply(lambda x: ' '.join([word for word in x.split() if word not in (stop)]))

In [11]:
df = df.drop(['id'], axis = 1)

In [12]:
df.head()


,tweet,label
0,The CDC currently reports 99031 deaths. In gen...,real
1,States reported 1121 deaths small rise last Tu...,real
2,Politically Correct Woman (Almost) Uses Pandem...,fake
3,#IndiaFightsCorona: We 1524 #COVID testing lab...,real
4,Populous states generate large case counts loo...,real


In [13]:
import string

In [14]:
punc = set(string.punctuation)

In [15]:
def remove_punc(st):
  for char in st:
    if char in punc:
      st = st.replace(char,'')
  return st

In [16]:
df['tweet'] = df['tweet'].apply(lambda x: remove_punc(x))
df.head()

,tweet,label
0,The CDC currently reports 99031 deaths In gene...,real
1,States reported 1121 deaths small rise last Tu...,real
2,Politically Correct Woman Almost Uses Pandemic...,fake
3,IndiaFightsCorona We 1524 COVID testing labora...,real
4,Populous states generate large case counts loo...,real


In [17]:
# Remove link to the source
def remove_links(tweet):
  indx = tweet.find('http')
  return tweet[:indx]

df['tweet'] = df['tweet'].apply(lambda x: remove_links(x))

In [18]:
tweets = df['tweet']
tweets = list(tweets)

In [19]:
# Tokenization 
nltk.download('punkt')
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [20]:
tokenized = []
for tweet in tweets:
  tokens = word_tokenize(tweet)
  tokenized.append(tokens)

In [21]:
# lemmatizatizer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Unzipping corpora/wordnet.zip.


True

In [22]:
lemmatizer = WordNetLemmatizer()
text = []
for i in range(len(tokenized)):
  for j in range(len(tokenized[i])):
    tokenized[i][j] =  lemmatizer.lemmatize(tokenized[i][j])
  
  text.append(" ".join(tokenized[i])) 

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
vectorizer = TfidfVectorizer()

In [25]:
vectorizer.fit(text)

TfidfVectorizer()

In [26]:
vector = vectorizer.fit_transform(text)
df1 = pd.DataFrame(vector[10].T.todense(), index = vectorizer.get_feature_names(), columns=["TF-IDF"])
df1 = df1.sort_values('TF-IDF', ascending=False)
print (df1.head())

               TF-IDF
simple       0.358774
precaution   0.358774
respiratory  0.327052
illness      0.298611
prevent      0.265501


/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


In [27]:
accuracy_scores = []

In [28]:
tf_vector = TfidfVectorizer(sublinear_tf=True)
tf_vector.fit(text)

X_text = tf_vector.transform(text)
y_values = np.array(df['label'].ravel())

In [29]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(y_values)
le.transform(y_values)

array([1, 1, 0, ..., 0, 0, 1])

In [39]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_text, y_values, test_size=0.3, random_state=42)

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)
y_predict = model.predict(X_test)

In [40]:
from sklearn.metrics import accuracy_score
score=accuracy_score(y_test,y_predict)
print(f'Accuracy: {round(score*100,2)}%')

Accuracy: 91.28%


In [34]:
from sklearn.metrics import classification_report

In [41]:
print(classification_report(y_test, y_predict))

              precision    recall  f1-score   support

        fake       0.89      0.93      0.91       920
        real       0.93      0.90      0.92      1006

    accuracy                           0.91      1926
   macro avg       0.91      0.91      0.91      1926
weighted avg       0.91      0.91      0.91      1926



In [42]:
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(text)
X = X.toarray()
y = df['label'].values

kf = KFold(n_splits=10, random_state=None, shuffle=True)
total_acc = 0
for train_index, test_index in kf.split(X):
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y[train_index], y[test_index]
  model = LogisticRegression()
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  accuracy = accuracy_score(y_test, y_pred)
  print(f'Accuracy Score of Logistic Regression: {round(accuracy*100,2)}%')
  total_acc += accuracy
avg_accuracy = total_acc / 10
accuracy_scores.append(avg_accuracy)
print(f'Average accuracy of the model: {round(avg_accuracy*100,2)}%')

Accuracy Score of Logistic Regression: 92.21%
Accuracy Score of Logistic Regression: 90.65%
Accuracy Score of Logistic Regression: 91.59%
Accuracy Score of Logistic Regression: 93.61%
Accuracy Score of Logistic Regression: 90.5%
Accuracy Score of Logistic Regression: 92.21%
Accuracy Score of Logistic Regression: 93.3%
Accuracy Score of Logistic Regression: 92.83%
Accuracy Score of Logistic Regression: 92.37%
Accuracy Score of Logistic Regression: 91.59%
Average accuracy of the model: 92.09%


In [44]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

        fake       0.90      0.93      0.91       296
        real       0.93      0.91      0.92       346

    accuracy                           0.92       642
   macro avg       0.91      0.92      0.92       642
weighted avg       0.92      0.92      0.92       642



### Project_3

In [46]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(text)
X = X.toarray()

In [47]:
y = df['label'].values

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=True, test_size=0.3, random_state=21)

In [64]:
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

In [65]:
print("Accuracy score with training data",clf.score(X_train, y_train)*100)

Accuracy score with training data 96.03916332888296


In [66]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10, random_state=None, shuffle=True)
total_acc = 0
for train_index, test_index in kf.split(X):
  X_train, X_test = X[train_index], X[test_index]
  y_train, y_test = y[train_index], y[test_index]
  clf = MultinomialNB()
  clf.fit(X_train, y_train)
  train_acc = clf.score(X_train, y_train)
  test_acc = clf.score(X_test, y_test)
  print(f'Accuracy Score of train set: {round(train_acc*100,2)}%')
  print(f'Accuracy Score of test set: {round(test_acc*100,2)}%')
  total_acc += test_acc
avg_accuracy = total_acc / 10
accuracy_scores.append(avg_accuracy)
print(f'Average test accuracy of the model: {round(avg_accuracy*100,2)}%')

Accuracy Score of train set: 95.67%
Accuracy Score of test set: 92.06%
Accuracy Score of train set: 95.57%
Accuracy Score of test set: 90.5%
Accuracy Score of train set: 95.86%
Accuracy Score of test set: 90.19%
Accuracy Score of train set: 95.4%
Accuracy Score of test set: 92.68%
Accuracy Score of train set: 95.76%
Accuracy Score of test set: 90.97%
Accuracy Score of train set: 95.59%
Accuracy Score of test set: 91.12%
Accuracy Score of train set: 95.86%
Accuracy Score of test set: 91.12%
Accuracy Score of train set: 95.73%
Accuracy Score of test set: 91.59%
Accuracy Score of train set: 95.71%
Accuracy Score of test set: 90.81%
Accuracy Score of train set: 95.6%
Accuracy Score of test set: 93.15%
Average test accuracy of the model: 91.42%


In [68]:
# Chek sentences : fake or real
def check_tweet(tweet):      
  vectorized_sentence = vectorizer.transform([tweet]).toarray()
  return clf.predict(vectorized_sentence)[0]

In [71]:
for tweet in text[:10]:
  print(tweet[:100])
  print(check_tweet(tweet))

The CDC currently report 99031 death In general discrepancy death count different source small expli
real
States reported 1121 death small rise last Tuesday Southern state reported 640 death
real
Politically Correct Woman Almost Uses Pandemic Excuse Not Reuse Plastic Bag
fake
IndiaFightsCorona We 1524 COVID testing laboratory India 25th August 2020 36827520 test done ProfBha
real
Populous state generate large case count look new case per million today 9 smaller state showing cas
real
Covid Act Now found on average person Illinois COVID19 infecting 111 people Data show infection grow
real
If tested positive COVID19 symptom stay home away people Learn CDC ’ s recommendation around others 
real
Obama Calls Trump ’ s Coronavirus Response A Chaotic Disaster
fake
Clearly Obama administration leave kind game plan something like this
fake
Retraction—Hydroxychloroquine chloroquine without macrolide treatment COVID19 multinational registry
fake


### PyTorch + LSTM

In [20]:
labels = (df.label == 'real').astype(int).to_list()

In [21]:
token_lists = [word_tokenize(text.lower()) for text in df.tweet]
max_len = len(max(token_lists, key=len))
max_len

1592

In [22]:
from collections import Counter
fd = Counter([len(tokens) for tokens in token_lists])

In [23]:
fd.most_common(10)

[(20, 179),
 (25, 174),
 (22, 170),
 (18, 170),
 (19, 167),
 (21, 167),
 (16, 162),
 (15, 161),
 (17, 161),
 (23, 157)]

In [24]:
def get_word_embedding(tokens, max_len):
    result = []
    for i in range(max_len):
        if i < len(tokens):
            word = tokens[i]
            if word in model_tweets.wv:
                result.append(model_tweets.wv[word])
            else:
                result.append(np.zeros(300))
        else:
            result.append(np.zeros(300))
    return result

In [25]:
from tqdm import tqdm
features = [get_word_embedding(text, 200) for text in tqdm(token_lists)]

100%|██████████| 6420/6420 [00:03<00:00, 1809.30it/s]


In [26]:
X_train, X_test, y_train, y_test = train_test_split(features, labels, test_size=0.33)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim


In [28]:
len(features[0][0])

300

In [29]:
len(X_train[0])

200

In [30]:
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()
        self.lstm = nn.LSTM(300, 100)
        self.out = nn.Linear(100, 1)

    def forward(self, x):
        embeddings, (shortterm, longterm) = self.lstm(x.transpose(0, 1))
        prediction = torch.sigmoid(self.out(longterm))
        return prediction


net = Net()
print(net)

Net(
  (lstm): LSTM(300, 100)
  (out): Linear(in_features=100, out_features=1, bias=True)
)


In [31]:
in_data = torch.tensor(X_train).float()
targets = torch.tensor(y_train).float()

/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  ../torch/csrc/utils/tensor_new.cpp:201.)
  """Entry point for launching an IPython kernel.


In [32]:
in_data.shape

torch.Size([4301, 200, 300])

In [40]:
optimizer = optim.SGD(net.parameters(),  lr=0.05)
criterion = nn.BCELoss()
# criterion = nn.Sigmoid()

In [41]:
def train_one_epoch(in_data, targets, batch_size=16):
    for i in tqdm(range(0, in_data.shape[0], batch_size)):
        batch_x = in_data[i:i + batch_size]
        batch_y = targets[i:i + batch_size]
        optimizer.zero_grad()
        output = net(batch_x)
        loss = criterion(output.reshape(-1), batch_y)
        loss.backward()
        optimizer.step()
    print(loss)

In [42]:
train_one_epoch(in_data, targets)

100%|██████████| 269/269 [03:23<00:00,  1.32it/s]

tensor(0.6906, grad_fn=<BinaryCrossEntropyBackward0>)


In [43]:
#result
in_data_test = torch.tensor(X_test).float()
targets_test = torch.tensor(y_test).float()

In [44]:
with torch.no_grad():
    output = net(in_data_test).reshape(-1)

In [45]:
result = (output > 0.5) == targets_test

In [46]:
result.sum().item() / len(result)

0.5077866918357716

### attempt_1

In [48]:
train_one_epoch(in_data, targets,batch_size=16)

100%|██████████| 269/269 [03:24<00:00,  1.31it/s]

tensor(0.6905, grad_fn=<BinaryCrossEntropyBackward0>)


In [ ]:
for i in tqdm(range(0,10)):
     batch_x = in_data[i:i + 100]
     batch_y = targets[i:i + 100]
     optimizer.zero_grad()
     output = net(batch_x)
     loss = criterion(output.reshape(-1), batch_y)
     loss.backward()
     optimizer.step()
     print(loss)

  0%|          | 0/10 [00:00<?, ?it/s]

In [73]:
#result
in_data_test_1 = torch.tensor(X_test).float()
targets_test_1 = torch.tensor(y_test).float()

In [74]:
with torch.no_grad():
    output = net(in_data_test_1).reshape(-1)

In [75]:
result = (output > 0.5) == targets_test_1

In [76]:
result.sum().item() / len(result)

0.4922133081642284

In [49]:
in_data_test = torch.tensor(X_test).float()
targets_test = torch.tensor(y_test).float()

In [50]:
with torch.no_grad():
    output = net(in_data_test).reshape(-1)

In [51]:
result = (output > 0.5) == targets_test

In [52]:
result.sum().item() / len(result)

0.5077866918357716